# Clase 1 – Manejo de Valores Perdidos y Outliers
### Limpieza y preparación de datos

**Temas:** Qué es un valor perdido · `isnull()` / `isna()` · `dropna()` · `fillna()` · `ffill()` / `bfill()` · Imputación condicional

---

In [ ]:
import pandas as pd
import numpy as np
print(f'pandas {pd.__version__} cargado ✅')

---
## Slide 3 – Desafío Inicial: Base de Datos de Pacientes Respiratorios

Columnas con valores faltantes: `edad`, `frecuencia_cardiaca`, `diagnostico`.
Columna con outliers evidentes: `temperatura_c` (valores como 75°C o −10°C).

**Archivo:** `pacientes_respiratorios.csv`

In [ ]:
# ── Slide 3: Carga del desafío inicial ───────────────────────────────────────

import pandas as pd
df = pd.read_csv('pacientes_respiratorios.csv')
print(f'Archivo cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(df.head())

---
## Slide 8 – Identificación de Valores Perdidos: `isnull()` y `notnull()`

In [ ]:
# ── Slide 8: isnull() y notnull() ────────────────────────────────────────────

df.isnull()   # Devuelve una tabla booleana
df.notnull()  # Lo inverso

---
## Slide 9 – `isna()` y conteo de nulos por columna

In [ ]:
# ── Slide 9: isna().sum() — cuenta los nulos por columna ─────────────────────

df.isna().sum()

In [ ]:
# ── Slide 9: Ejemplo aplicado ────────────────────────────────────────────────

import pandas as pd
df = pd.DataFrame({'Nombre': ['Ana', 'Luis', None, 'Marta'],
                   'Edad':   [28, None, 35, 40],
                   'Ciudad': ['Santiago', 'Valparaíso', 'Santiago', None]})
print(df.isnull().sum())

---
## Slide 10 – Detección de Filas con Valores Perdidos

In [ ]:
# ── Slide 10: Filas con al menos un valor perdido ────────────────────────────

df[df.isnull().any(axis=1)]

---
## Slide 12 – Eliminación de Datos Faltantes: `dropna()`

In [ ]:
# ── Slide 12: dropna() — elimina filas con al menos un nulo ──────────────────

df.dropna()         # elimina filas con cualquier nulo
df.dropna(axis=1)   # elimina columnas con valores nulos

---
## Slide 16 – Imputación con Valores Estadísticos

In [ ]:
# ── Slide 16: fillna() con media y mediana (variables numéricas) ──────────────

df['Edad'].fillna(df['Edad'].mean(),   inplace=True)   # media
df['Edad'].fillna(df['Edad'].median(), inplace=True)   # mediana

In [ ]:
# ── Slide 16: fillna() con moda (variables categóricas) ──────────────────────

df['Ciudad'].fillna(df['Ciudad'].mode()[0], inplace=True)

---
## Slide 17 – Imputación con Valor Fijo

In [ ]:
# ── Slide 17: fillna() con valor fijo ────────────────────────────────────────

df['Diagnóstico'].fillna('No informado', inplace=True)

---
## Slide 18 – Imputación por Propagación (`ffill` / `bfill`)

In [ ]:
# ── Slide 18: ffill() — propagación hacia adelante ───────────────────────────
# Se usa con registros ordenados (como series temporales)

df.ffill()

In [ ]:
# ── Slide 18: bfill() — propagación hacia atrás ──────────────────────────────

df.bfill()

---
## Slide 19 – Imputación Condicional (avanzada)

In [ ]:
# ── Slide 19: fillna() con media por grupo (groupby + transform) ──────────────
# Reemplaza valores faltantes con la media del grupo correspondiente

df['Edad'] = df.groupby('Sexo')['Edad'].transform(lambda x: x.fillna(x.mean()))

---
## Slides 21–24 – Actividad Guiada: Registros de Pacientes

**Objetivo (Slide 22):** Detectar y tratar valores perdidos evaluando distintas estrategias de imputación.

In [ ]:
# ── Slide 23: Crear el DataFrame base ────────────────────────────────────────

import pandas as pd
df = pd.DataFrame({
    'Paciente': ['Ana', 'Luis', 'Marcela', 'Diego', 'Camila'],
    'Edad': [29, 41, None, 35, 38],
    'Frecuencia Cardíaca': [78, 85, 88, None, 79],
    'Diagnóstico': ['Gripe', None, 'COVID-19', 'Asma', 'Gripe']
})
# Identificar valores perdidos:
print(df.isnull().sum())

In [ ]:
# ── Slide 24: Eliminar filas con nulos (solo para observar impacto) ───────────

df_sin_nulos = df.dropna()
# Imputar valores perdidos:
# - Edad con media:
df['Edad'].fillna(df['Edad'].mean(), inplace=True)
# - Frecuencia con mediana:
df['Frecuencia Cardíaca'].fillna(df['Frecuencia Cardíaca'].median(), inplace=True)
# - Diagnóstico con valor fijo:
df['Diagnóstico'].fillna('No informado', inplace=True)

In [ ]:
# ── Slide 24: Comparar antes y después ───────────────────────────────────────

print('=== Sin imputar (solo dropna) ===')
print(df_sin_nulos)
print(f'\nRegistros conservados con dropna: {len(df_sin_nulos)} de 5')
print()
print('=== Con imputación ===')
print(df)
print(f'\nNulos restantes después de imputar:')
print(df.isnull().sum())

---
## Actividad Guiada ampliada — `pacientes_respiratorios.csv` (n=200)

Aplicamos la misma secuencia al dataset real del desafío inicial.

In [ ]:
# Carga y diagnóstico inicial
df_pac = pd.read_csv('pacientes_respiratorios.csv')

print('=== Diagnóstico de nulos ===')
nulos = df_pac.isnull().sum()
pct   = (nulos / len(df_pac) * 100).round(1)
print(pd.DataFrame({'nulos': nulos, '% del total': pct}).query('nulos > 0'))

print('\n=== Filas con al menos un nulo ===')
print(df_pac[df_pac.isnull().any(axis=1)].shape[0], 'registros afectados')

In [ ]:
# Outliers en temperatura (Slide 3: valores como 75°C o −10°C)
outliers_temp = df_pac[(df_pac['temperatura_c'] < 35) | (df_pac['temperatura_c'] > 41)]
print(f'Outliers en temperatura_c: {len(outliers_temp)}')
print(outliers_temp[['id_paciente','temperatura_c']])

In [ ]:
# Imputación siguiendo criterios del Slide 20
df_limpio = df_pac.copy()

# edad → media (numérica, distribución aproximadamente normal)
df_limpio['edad'].fillna(df_limpio['edad'].mean(), inplace=True)

# frecuencia_cardiaca → mediana (más robusta ante posibles extremos)
df_limpio['frecuencia_cardiaca'].fillna(df_limpio['frecuencia_cardiaca'].median(), inplace=True)

# diagnostico → valor fijo (categórica, ausencia tiene significado)
df_limpio['diagnostico'].fillna('No informado', inplace=True)

# temperatura outliers → reemplazar con mediana (valores físicamente imposibles)
mediana_temp = df_limpio[(df_limpio['temperatura_c'] >= 35) & (df_limpio['temperatura_c'] <= 41)]['temperatura_c'].median()
df_limpio.loc[(df_limpio['temperatura_c'] < 35) | (df_limpio['temperatura_c'] > 41), 'temperatura_c'] = mediana_temp

print('=== Nulos después de imputar ===')
print(df_limpio.isnull().sum())
print(f'\nOutliers temperatura restantes: {((df_limpio["temperatura_c"] < 35) | (df_limpio["temperatura_c"] > 41)).sum()}')

---
## Slides 25–29 – Actividad Autónoma: Registros Educativos

**Contexto (Slide 26):** Base de datos de estudiantes en capacitación técnica con valores faltantes en edad, modalidad y calificación.

---
**Completa las celdas marcadas con `# 📝 TU CÓDIGO AQUÍ`**

In [ ]:
# ── Slide 27: Crear el DataFrame de estudiantes ───────────────────────────────

data = {'Estudiante': ['Pedro', 'Laura', 'Ignacio', 'Fernanda', 'Javiera'],
        'Edad': [22, None, 27, 25, None],
        'Modalidad': ['Online', 'Presencial', None, 'Online', 'Presencial'],
        'Calificación': [5.5, 6.2, None, 5.8, 6.0]}
df = pd.DataFrame(data)

In [ ]:
# ── Slide 28, Paso 1: Detectar y cuantificar los valores perdidos ─────────────

print(df.isnull().sum())   # Revisión por columna

# 📝 TU CÓDIGO AQUÍ
# Muestra también las filas con nulos: df[df.isnull().any(axis=1)]

In [ ]:
# ── Slide 28, Paso 2a: Imputar Edad con promedio ──────────────────────────────

# 📝 TU CÓDIGO AQUÍ
# df['Edad'].fillna(..., inplace=True)

In [ ]:
# ── Slide 28, Paso 2b: Imputar Modalidad con valor más frecuente (moda) ───────

# 📝 TU CÓDIGO AQUÍ
# df['Modalidad'].fillna(..., inplace=True)

In [ ]:
# ── Slide 28, Paso 2c: Imputar Calificación con mediana ───────────────────────

# 📝 TU CÓDIGO AQUÍ
# df['Calificación'].fillna(..., inplace=True)

In [ ]:
# ── Slide 29: Verificar y guardar el DataFrame limpio ─────────────────────────

# 📝 TU CÓDIGO AQUÍ
# print(df)                         # DataFrame final
# print(df.isnull().sum())          # ¿Quedan nulos?
# df.to_csv('estudiantes_limpio.csv', index=False)

### Tabla de decisiones de imputación (Slide 29)

| Variable | Método elegido | Justificación |
|----------|----------------|---------------|
| Edad | Promedio | Variable numérica con variación moderada |
| Modalidad | Moda (más frecuente) | Variable categórica con baja dispersión |
| Calificación | Mediana | Minimiza el impacto de valores extremos |